In [1]:
# from google.colab import drive
# # mount google drive
# drive.mount('/content/drive/')

In [2]:
# !pip install jsonargparse==4.28.0 lightning==2.2.3 seaborn==0.13.2 tabulate==0.9.0 termcolor==2.4.0 torch==2.2.2 torchmetrics==1.3.2 torchvision==0.17.2 wandb==0.16.6 wget==3.2 tqdm==4.66.2

In [3]:
# cd /content/drive/MyDrive/hyu/aue8088-pa1

In [4]:
# PyTorch & Pytorch Lightning
from lightning.pytorch.loggers.wandb import WandbLogger
from lightning.pytorch.callbacks import LearningRateMonitor, ModelCheckpoint
from lightning import Trainer
import torch

# Custom packages
from src.dataset import TinyImageNetDatasetModule
from src.network import SimpleClassifier
import src.config as cfg

torch.set_float32_matmul_precision('medium')

In [5]:
model = SimpleClassifier(
    model_name = cfg.MODEL_NAME,
    num_classes = cfg.NUM_CLASSES,
    optimizer_params = cfg.OPTIMIZER_PARAMS,
    scheduler_params = cfg.SCHEDULER_PARAMS,
)

datamodule = TinyImageNetDatasetModule(
    batch_size = cfg.BATCH_SIZE,
)

wandb_logger = WandbLogger(
    project = cfg.WANDB_PROJECT,
    save_dir = cfg.WANDB_SAVE_DIR,
    entity = cfg.WANDB_ENTITY,
    name = cfg.WANDB_NAME,
)

trainer = Trainer(
    accelerator = cfg.ACCELERATOR,
    devices = cfg.DEVICES,
    precision = cfg.PRECISION_STR,
    max_epochs = cfg.NUM_EPOCHS,
    check_val_every_n_epoch = cfg.VAL_EVERY_N_EPOCH,
    logger = wandb_logger,
    callbacks = [
        LearningRateMonitor(logging_interval='epoch'),
        ModelCheckpoint(save_top_k=1, monitor='accuracy/val', mode='max'),
    ],
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [6]:
# Takes time at first (download dataset)
trainer.fit(model, datamodule=datamodule)

Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: bo_oseng. Use `wandb login --relogin` to force relogin


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]

  | Name     | Type             | Params
----------------------------------------------
0 | model    | ResNet           | 11.3 M
1 | loss_fn  | CrossEntropyLoss | 0     
2 | accuracy | MyAccuracy       | 0     
3 | f1score  | MyF1Score        | 0     
----------------------------------------------
11.3 M    Trainable params
0         Non-trainable params
11.3 M    Total params
45.116    Total estimated model params size (MB)


Sanity Checking: |          | 0/? [00:00<?, ?it/s][Val]	 root dir: datasets/tiny-imagenet-200/val	 | # of samples: 10,000
[Train]	 root dir: datasets/tiny-imagenet-200/train	 | # of samples: 100,000


/home/user/.conda/envs/aue8088/lib/python3.8/site-packages/lightning/pytorch/loops/fit_loop.py:298: The number of training batches (3) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.


Training: |          | 0/? [00:00<?, ?it/s]ACCELERATOR: gpu
BATCH_SIZE: 40960
DATASET_ROOT_PATH: datasets/
DEVICES: [0]
IMAGE_FLIP_PROB: 0.5
IMAGE_MEAN: [0.4802, 0.4481, 0.3975]
IMAGE_NUM_CROPS: 64
IMAGE_PAD_CROPS: 4
IMAGE_ROTATION: 20
IMAGE_STD: [0.2302, 0.2265, 0.2262]
MODEL_NAME: resnet18
NUM_CLASSES: 200
NUM_EPOCHS: 40
NUM_WORKERS: 8
OPTIMIZER_PARAMS: {'type': 'SGD', 'lr': 0.005, 'momentum': 0.9}
PRECISION_STR: 32-true
SCHEDULER_PARAMS: {'type': 'MultiStepLR', 'milestones': [30, 35], 'gamma': 0.2}
VAL_EVERY_N_EPOCH: 1
WANDB_ENTITY: None
WANDB_IMG_LOG_FREQ: 50
WANDB_NAME: resnet18-B40960-SGD-MultiStepLR5.0E-03
WANDB_PROJECT: aue8088-pa1
WANDB_SAVE_DIR: wandb/
Epoch 39: 100%|██████████| 3/3 [00:28<00:00,  0.10it/s, v_num=dvph, loss/val=5.040, accuracy/val=0.030, f1score/val=0.0169, loss/train=5.040, accuracy/train=0.0324, f1score/train=0.0193]   

`Trainer.fit` stopped: `max_epochs=40` reached.


Epoch 39: 100%|██████████| 3/3 [00:28<00:00,  0.10it/s, v_num=dvph, loss/val=5.040, accuracy/val=0.030, f1score/val=0.0169, loss/train=5.040, accuracy/train=0.0324, f1score/train=0.0193]


In [7]:
trainer.validate(ckpt_path='best', datamodule=datamodule)

Restoring states from the checkpoint path at wandb/aue8088-pa1/ukawdvph/checkpoints/epoch=39-step=120.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]
Loaded model weights from the checkpoint at wandb/aue8088-pa1/ukawdvph/checkpoints/epoch=39-step=120.ckpt


[Val]	 root dir: datasets/tiny-imagenet-200/val	 | # of samples: 10,000
Validation DataLoader 0: 100%|██████████| 1/1 [00:00<00:00,  4.37it/s]
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
     Validate metric           DataLoader 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
      accuracy/val         0.029999999329447746
       f1score/val         0.016908785328269005
        loss/val             5.041152000427246
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────


[{'loss/val': 5.041152000427246,
  'accuracy/val': 0.029999999329447746,
  'f1score/val': 0.016908785328269005}]